In [ ]:
import numpy as np
import random

from scipy.stats import linregress

import plotly.graph_objects as go

### Bachelier model

Yahooquery documentation:<br>
https://yahooquery.dpguthrie.com/<br>
https://yahooquery.dpguthrie.com/guide/ticker/historical/

In [ ]:
from yahooquery import Ticker

In [ ]:
def bm_simulations(n_paths, granularity, max_time):
    n_steps = granularity * max_time
    
    paths = []
    for i in range(n_paths):
        xs = [0] + random.choices([-1, 1], weights = [0.5, 0.5], k = n_steps)
        path = np.cumsum(xs)/np.sqrt(granularity)
        paths.append(path)
        
    return np.array(paths)

To check how models works choose and analyse trajectory Feb 19, 2019 - Feb 21, 2020

In [ ]:
cola = Ticker('KO')
cola_data = cola.history(start = '2019-02-19', end = '2020-02-21', interval = '1d')
cola_data
t = np.array(cola_data.index.get_level_values(level=1))
prices = np.array(cola_data['close'])

In [ ]:
len(t)

<H3>Bachelier model</H3> - first serious model in mathematical finance. It assumes that stock prices $(S_t)_{t\geq 0}$ satisfy equation
$$S_t=S_0+\mu t+\sigma W_t$$
$\mu$ is called <i>trend</i>, $\sigma$ is called <i>volatility</i>.

Estimate $\mu$ via linear regression $y=\mu x+\varepsilon$ with $x=t,\,y=S_t=prices[t]$

Seems that volatility is a bit high. Try $\chi^2$ approach: relative errors 
$$
\chi^2_{st} = \sum_{i=1}^k \frac{(Obs_i-Exp_i)^2}{Exp_i}
$$
credit: @Alexander Polyakov

<H3>Confidence interval for future price $S_T$</H3>

In [ ]:
def bachelier_params(train_prices):
    s0 = train_prices[0]
#     t = np.array(range(len(train_prices)))
    t = np.arange(len(train_prices))
                 
    ans = linregress(t, train_prices)
    mu = ans.slope
                 
    devs = (train_prices - mu*t - s0)**2/(s0 + mu*t)
    sigma = np.sqrt(np.mean(devs))
    
    return [mu, sigma]

In [ ]:
# result of the previous lesson
bachelier_params(prices)

Divide sample of prices to training and test tests. Estimate parameters on training set and then predict the model on test set.

Division of initial sample by training set ($\approx 80\%$) and test set ($\approx 20\%$)

In [ ]:
t_numer = np.arange(len(t))
train_days = int(0.8*len(t))
train_days
params = bachelier_params(prices[:train_days])
params

In [ ]:
test_days = len(t) - train_days 
test_prices = prices[train_days:]
st_0 = prices[train_days]
st_0

$S_t = S_{t_0}+\widehat{\mu} (t-t_0) +\widehat\sigma W_{t-t_0},\,t\ge t_0$ on a test set.

In [ ]:
train_days

In [ ]:
N = 8
T = test_days - 1
gran_factor = 10
W = bm_simulations(N, gran_factor,T)

In [ ]:
len(t)

* Draw original prices trajectory on a train set and test set
* Draw prices simulations on a test set and visually compare the quality of prediction

In [ ]:
t_new = np.linspace(train_days, len(t)-1, gran_factor*T+1)
len(t_new) == len(W[0])
mu = params[0]
sigma = params[1]
st = []
for i in range(0, N):
    st_i = st_0 + mu*(t_new - t_numer[train_days]) + sigma*W[i]
    st.append(st_i)

In [ ]:
plot = go.Figure()

graph = go.Scatter(x = t_numer, y = prices, name = 'Prices')
plot.add_trace(graph)
for i in range(0, N):
    graph1 = go.Scatter(x = t_new, y = st[i], name = f'BM with drift {i}')
    plot.add_trace(graph1)
plot.show()

Quality of prediction is metrics e.g. RMSE:
$$
RMSE[path] = \sqrt{\frac{1}{n}\sum_{i=1}^n (S_{t_i}-\widehat{S_{t_i}})^2}
$$
$$
RMSE = \frac{1}{N}\sum_{path=0}^N RMSE[path]
$$
where times $t_i$ are taken on a test set

In [ ]:
# np.array(range(0, len(t_new), gran_factor))
# len(st[0])

In [ ]:
prices_test = prices[train_days:]
rmse_paths = []
for i in range(0, N):
    rmse_path =[]
    for j in range(0, len(t_new), gran_factor):
        rmse_path.append(st[i][j])
    rmse_paths.append(np.array(rmse_path))

deltas = (np.array(rmse_paths) - prices_test)**2
# collection RMSE[path]
deltas_mean = [np.sqrt(np.mean(delta)) for delta in deltas]
deltas_mean
rmse = np.mean(deltas_mean)
rmse